In [1]:
import pandas as pd

In [ ]:
DATA_DIR = 'data'
df = pd.read_parquet(f'{DATA_DIR}/hyperkalemia2.parquet')

In [3]:
hyperkalemia_codes = ["LOINC/LG7931-1", "LOINC/LP386618-5", "LOINC/LG10990-6", "LOINC/2823-3"]
# remove all df rows that have a code in hypoglycemia_codes
print(f"Before removing hyperkalemia codes: {len(df)} rows")
df = df[~df['code'].isin(hyperkalemia_codes)]
print(f"After removing hyperkalemia codes: {len(df)} rows")

Before removing hyperkalemia codes: 12177732 rows
After removing hyperkalemia codes: 12113470 rows


In [35]:
labels = pd.read_parquet('hyperkalemia_label2.parquet')
# rename 'value' to 'label'
labels.rename(columns={'value': 'label'}, inplace=True)
# create a new column 'Time' in labels, which is Sample_time filled with prediction_time if na
labels['Time'] = labels['Sample_time'].fillna(labels['prediction_time'])
labels.head()

,patient_id,prediction_time,label,value_multi,start,Sample_time,Time
0,115967149,2021-09-16 19:21:00,True,3,2021-07-07 13:47:00,NaT,2021-09-16 19:21:00
1,115967186,2014-05-29 04:59:00,True,3,2014-05-23 12:01:00,NaT,2014-05-29 04:59:00
2,115967217,2002-06-30 20:59:00,True,3,2002-06-28 19:55:00,NaT,2002-06-30 20:59:00
3,115967241,2019-07-01 08:32:00,True,3,2019-06-20 23:59:00,NaT,2019-07-01 08:32:00
4,115967262,2017-02-24 13:01:00,True,3,2016-11-02 04:45:00,NaT,2017-02-24 13:01:00


In [6]:
# find positive fraction 
pos_labels = labels[labels['label'] == True]
print("Positive fraction: ", len(pos_labels) / len(labels))

Positive fraction:  0.04033755274261604


In [7]:
# find time difference between 'Time' and 'start' columns
labels['Time'] = pd.to_datetime(labels['Time'])
labels['visit_start'] = pd.to_datetime(labels['start'])
labels['visit_end'] = labels['Time'] - pd.Timedelta(hours=1)
labels.head()

,patient_id,prediction_time,label,value_multi,start,Sample_time,Time,visit_start,visit_end
0,115967149,2021-09-16 19:21:00,True,3,2021-07-07 13:47:00,NaT,2021-09-16 19:21:00,2021-07-07 13:47:00,2021-09-16 18:21:00
1,115967186,2014-05-29 04:59:00,True,3,2014-05-23 12:01:00,NaT,2014-05-29 04:59:00,2014-05-23 12:01:00,2014-05-29 03:59:00
2,115967217,2002-06-30 20:59:00,True,3,2002-06-28 19:55:00,NaT,2002-06-30 20:59:00,2002-06-28 19:55:00,2002-06-30 19:59:00
3,115967241,2019-07-01 08:32:00,True,3,2019-06-20 23:59:00,NaT,2019-07-01 08:32:00,2019-06-20 23:59:00,2019-07-01 07:32:00
4,115967262,2017-02-24 13:01:00,True,3,2016-11-02 04:45:00,NaT,2017-02-24 13:01:00,2016-11-02 04:45:00,2017-02-24 12:01:00


In [8]:
# drop all columns in df except patient_id,start_time, code, value, 
df = df[['patient_id', 'start', 'code', 'value']]
# rename 'start' to 'event_time'
df.rename(columns={'start': 'event_time'}, inplace=True)
# drop all columns in labels except patient_id, visit_start, label, visit_end
labels = labels[['patient_id', 'visit_start', 'label', 'visit_end']]

In [9]:
# merge my_df and my_labels on patient_id
merged_df = df.merge(labels, on='patient_id', how='inner')
# filter my_df to only include rows where start_time is between start and end
merged_df = merged_df[(merged_df['event_time'] >= merged_df['visit_start']) & (merged_df['event_time'] <= merged_df['visit_end'])]

In [10]:
# keep only patient_id, event_time, code, value
merged_df = merged_df[['patient_id', 'event_time', 'code', 'value']]

In [11]:
# sort by patient_id, event_time
merged_df = merged_df.sort_values(by=['patient_id', 'event_time'])
# keep only the last 300 rows for each patient_id
merged_df = merged_df.groupby('patient_id').tail(500)

In [12]:
# for the `value` in merged_df, if the value can be interpreted as a float, convert it to float with 2 decimal places
def convert_to_float(value):
    try:
        return round(float(value), 2)
    except:
        return value
merged_df['value'] = merged_df['value'].apply(convert_to_float)

In [13]:
# compute the min event_time for each patient, and then create a new column `relative_time_mins` which is the difference between event_time and the min event_time for that patient
merged_df['event_time'] = pd.to_datetime(merged_df['event_time'])
merged_df['min_event_time'] = merged_df.groupby('patient_id')['event_time'].transform('min')
merged_df['relative_time_mins'] = (merged_df['event_time'] - merged_df['min_event_time']).dt.total_seconds() / 60.0
# drop the min_event_time column
merged_df.drop(columns=['min_event_time'], inplace=True)
merged_df

,patient_id,event_time,code,value,relative_time_mins
325,115967095,2008-10-10 06:05:00,LOINC/731-0,2.25,0.0
326,115967095,2008-10-10 06:05:00,LOINC/5905-5,2.9,0.0
327,115967095,2008-10-10 06:05:00,LOINC/713-8,0.4,0.0
328,115967095,2008-10-10 06:05:00,LOINC/4544-3,35.5,0.0
329,115967095,2008-10-10 06:05:00,LOINC/17861-6,6.9,0.0
...,...,...,...,...,...
12102612,121124989,2017-10-15 23:59:00,SNOMED/230057008,N,1900.0
12102613,121124989,2017-10-15 23:59:00,Medicare Specialty/A0,None,1900.0
12102614,121124989,2017-10-15 23:59:00,SNOMED/417662000,None,1900.0
12102615,121124989,2017-10-15 23:59:00,SNOMED/110483000,None,1900.0


In [ ]:
all_codes = pd.read_parquet('all_codes.parquet')

In [15]:
# join merged_df with all_codes on code
merged_df = merged_df.merge(all_codes, on='code', how='left')


In [16]:
# in 'description' column remove all occurrences of '#' or 'Bld Auto', or any text in parentheses eg '(foo)'
merged_df['description'] = merged_df['description'].str.replace('#', '', regex=False)
merged_df['description'] = merged_df['description'].str.replace('Bld Auto', '', regex=False)
merged_df['description'] = merged_df['description'].str.replace(r'\(.*?\)', '', regex=True)

In [17]:
# drop all nan values in the description column
merged_df = merged_df[merged_df['description'].notna()]
# strip all whitespace from the description column
merged_df['description'] = merged_df['description'].str.strip()

In [30]:
# find rows where description column contain 'potasium'
potassium_rows = merged_df[merged_df['description'].str.contains('potassium', case=False)]
potassium_rows



,patient_id,event_time,code,value,relative_time_mins,description,Text
715,115967098,1999-09-30 23:59:00,SNOMED/59573005,4.7,1064.0,Potassium measurement,Potassium measurement: 4.7
770,115967098,1999-10-01 23:59:00,SNOMED/59573005,4.9,2504.0,Potassium measurement,Potassium measurement: 4.9
837,115967098,1999-10-02 23:59:00,SNOMED/59573005,4.7,3944.0,Potassium measurement,Potassium measurement: 4.7
892,115967098,1999-10-03 23:59:00,SNOMED/59573005,5.6,5384.0,Potassium measurement,Potassium measurement: 5.6
952,115967098,1999-10-04 23:59:00,SNOMED/59573005,5.5,6824.0,Potassium measurement,Potassium measurement: 5.5
...,...,...,...,...,...,...,...
1121796,115973815,2011-07-31 01:04:00,RxNorm/1860466,None,149116.0,50 ML potassium chloride 0.4 MEQ/ML Injection,50 ML potassium chloride 0.4 MEQ/ML Injection
1122239,115973818,2013-11-19 23:59:00,RxNorm/1860466,None,1057.0,50 ML potassium chloride 0.4 MEQ/ML Injection,50 ML potassium chloride 0.4 MEQ/ML Injection
1122767,115973822,2014-11-26 00:51:00,LOINC/6298-4,3.9,18.0,Potassium Bld-sCnc,Potassium Bld-sCnc: 3.9
1123101,115973826,2014-01-19 07:01:00,LOINC/6298-4,3.9,20582.0,Potassium Bld-sCnc,Potassium Bld-sCnc: 3.9


In [18]:
# create text column by concatenating the description and value columns ONLY if the value is not nan
# else, just use the description column
merged_df['Text'] = merged_df.apply(lambda x: x['description'] + ': ' + str(x['value']) if pd.notna(x['value']) else x['description'], axis=1)

In [19]:
def _condense_notes_per_csn(group: pd.DataFrame) -> str:
    """Helper to condense text notes for a patient_id group."""
    # For rows with the same time_difference, we want to combine them into a single entry
    group = group.groupby('relative_time_mins').agg({
        'Text': lambda x: ";".join(x.astype(str))
    }).reset_index()

    return "\n".join(
        f"{row['relative_time_mins']} - {row['Text']}"
        for _, row in group.astype({'Text': str, 'relative_time_mins': str}).iterrows()
    )
# group by patient_id and relative_time_mins, and then condense the text notes
condensed_df = merged_df.groupby('patient_id').apply(_condense_notes_per_csn).reset_index()
condensed_df.rename(columns={0: 'Text'}, inplace=True)
condensed_df.head()

/tmp/ipykernel_1858710/3955256315.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  condensed_df = merged_df.groupby('patient_id').apply(_condense_notes_per_csn).reset_index()


,patient_id,Text
0,115967095,0.0 - Lymphocytes: 2.25;Monocytes NFr: 2.9;Eos...
1,115967096,0.0 - Heart rate: 87.0;Bdy temp measurement si...
2,115967097,0.0 - Transpl surg Note\n969120.0 - Transpl su...
3,115967098,0.0 - ALP SerPl-cCnc: 262.0;RBC: 3.6;Creat Ser...
4,115967099,313078.0 - Prog note\n325004.0 - Inhaled O2 fl...


In [20]:
demographics = pd.read_parquet('hyperkalemia_demo.parquet')
# keep only the patient_id and age columns
demographics = demographics[['patient_id', 'gender', 'age']]
# demo_text = 'Gender: ' + gender + ', Age: ' + age
demographics['DemoText'] = demographics.apply(lambda x: 'Gender' + ': ' + str(x['gender']) + ', ' + 'Age' + ': ' + str(x['age']), axis=1)
demographics = demographics[['patient_id', 'DemoText']]

In [21]:
# merge condensed_df and demographics on patient_id
# if patient_id is not in condensed_df, then do not include it in the merged_df
condensed_df = condensed_df.merge(demographics, on='patient_id', how='inner')

In [22]:
condensed_df.head()

,patient_id,Text,DemoText
0,115967095,0.0 - Lymphocytes: 2.25;Monocytes NFr: 2.9;Eos...,"Gender: F, Age: 74"
1,115967096,0.0 - Heart rate: 87.0;Bdy temp measurement si...,"Gender: F, Age: 68"
2,115967097,0.0 - Transpl surg Note\n969120.0 - Transpl su...,"Gender: F, Age: 49"
3,115967098,0.0 - ALP SerPl-cCnc: 262.0;RBC: 3.6;Creat Ser...,"Gender: F, Age: 53"
4,115967099,313078.0 - Prog note\n325004.0 - Inhaled O2 fl...,"Gender: F, Age: 65"


In [23]:
# now concatenate the DemoText and Text columns into a single column called 'Text'
condensed_df['Text'] = condensed_df['DemoText'] + '\n' + condensed_df['Text']

In [24]:
# now merge condensed_df with labels on patient_id
condensed_df = condensed_df.merge(labels[['patient_id', 'label']], on='patient_id', how='inner')

In [25]:
condensed_df.head()

,patient_id,Text,DemoText,label
0,115967095,"Gender: F, Age: 74\n0.0 - Lymphocytes: 2.25;Mo...","Gender: F, Age: 74",False
1,115967096,"Gender: F, Age: 68\n0.0 - Heart rate: 87.0;Bdy...","Gender: F, Age: 68",False
2,115967097,"Gender: F, Age: 49\n0.0 - Transpl surg Note\n9...","Gender: F, Age: 49",False
3,115967098,"Gender: F, Age: 53\n0.0 - ALP SerPl-cCnc: 262....","Gender: F, Age: 53",False
4,115967099,"Gender: F, Age: 65\n313078.0 - Prog note\n3250...","Gender: F, Age: 65",False


In [29]:
# rename label to Label
condensed_df.rename(columns={"label": "Label"}, inplace=True)

In [27]:
# save to llm_full_hypoglycemia.parquet
condensed_df.to_parquet('textified_hyperkalemia_full.parquet', index=False)

In [28]:
# also, split the data into val and test sets, in a 70-30 split
from sklearn.model_selection import train_test_split
val_df, test_df = train_test_split(condensed_df, test_size=0.3, random_state=1234)
val_df.to_parquet('textified_hyperkalemia_val.parquet', index=False)
test_df.to_parquet('textified_hyperkalemia_test.parquet', index=False)